# **Data importing and data standardization and correlation heat map**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import copy


train = pd.read_excel("BNT-Energy2f.xlsx")
X=train.iloc[:,:-1]
Y=train.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)


scalerX = StandardScaler().fit(X_train)
scalery = StandardScaler().fit(y_train.values.reshape(-1,1))
X_train1 = scalerX.transform(X_train)
y_train1 = scalery.transform(y_train.values.reshape(-1,1))
X_test1 = scalerX.transform(X_test)
y_test1 = scalery.transform(y_test.values.reshape(-1,1))
y_new_inverse = scalery.inverse_transform(y_test1.reshape(-1,1))
X_train2=copy.deepcopy(X_train1)
y_train2=copy.deepcopy(y_train1)
#print(y_new_inverse)
#train.corr()
#sns.heatmap(train.corr(), cmap="YlGnBu", annot=False)

import matplotlib.pyplot as plt
import seaborn as sns

corr = train.corr() #your dataframe

# figsize=(6, 6) control width and height
# dpi = 600, I
plt.figure(figsize=(10, 10),
           dpi = 600)

# parameter annot_kws={"size": 8} control corr values font size
sns.heatmap(corr, cmap="Blues", annot=False, annot_kws={"size": 2})

plt.tick_params(axis = 'x', labelsize = 6) # x font label size
plt.tick_params(axis = 'y', labelsize = 6) # y font label size

# **Model defining and training and best parameters with performance metric**

In [ ]:
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from xgboost import XGBRegressor
# from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

xgb_model=XGBRegressor()

search_space={'n_estimators':[100,200,300,400,500,600,800],'max_depth':[3,4,6,7,9],'gamma0':[0.01],'learning_rate':[0.001,0.01,0.05,0.1,0.5,1]}


In [ ]:
from sklearn.model_selection import GridSearchCV

GS=GridSearchCV(estimator=xgb_model,param_grid=search_space,scoring=["r2","neg_root_mean_squared_error"],refit="neg_root_mean_squared_error",cv=5,verbose=4)
GS.fit(X_train1,y_train1)

In [ ]:
print(GS.best_params_) # ND gamma0=0.01, learning_rate=0.01, max_depth=9, n_estimators=800;
print(GS.best_score_)
from sklearn.metrics import r2_score
model_xgb=XGBRegressor(n_estimators=800,max_depth=3,eta=0.01)
model_xgb.fit(X_train1,y_train1)
#y_pred=model_xgb.predict(X_test1)

#y_new_inverse = scalery.inverse_transform(y_pred.reshape(-1,1))
#Y_true=scalery.inverse_transform(y_test1)
#r2_score(Y_true,y_new_inverse)

{'gamma0': 0.01, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 800}
-0.35812038335964436


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eta=0.01, eval_metric=None,
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=800, n_jobs=None, ...)


# **Model Prediction and plot with error bars using bootstrapping**

In [ ]:
from sklearn.metrics import mean_squared_error
bootstrap_means=[]
bootstrap_means2=[]

N=500
for i in range(0,N):
  sample_index=np.random.choice(range(0,len(y_train1)),len(y_train1))

  X_samples=X_train1[sample_index]
  y_samples=y_train1[sample_index]
  print(i)

  model_xgb=XGBRegressor(n_estimators=800,max_depth=3,eta=0.01)

  model_xgb.fit(X_samples, y_samples)
  y_prediction=model_xgb.predict(X_test1)
  y_train2_prediction=model_xgb.predict(X_train2)
  bootstrap_means.append(scalery.inverse_transform(y_prediction.reshape(-1,1)))
  bootstrap_means2.append(scalery.inverse_transform(y_train2_prediction.reshape(-1,1)))

bootstrap_means_array=np.array(bootstrap_means)
bootstrap_means2_array=np.array(bootstrap_means2)

print(r2_score(scalery.inverse_transform(y_test1),bootstrap_means_array.mean(0)))
#print(mean_squared_error(scalery.inverse_transform(y_test1),bootstrap_means_array.mean(0)))
print(np.sqrt(mean_squared_error(scalery.inverse_transform(y_test1),bootstrap_means_array.mean(0))))
print(np.mean(np.abs(y_test1-bootstrap_means_array.mean(0))))

print(r2_score(scalery.inverse_transform(y_train2),bootstrap_means2_array.mean(0)))
#print(mean_squared_error(scalery.inverse_transform(y_train2),bootstrap_means2_array.mean(0)))
print(np.sqrt(mean_squared_error(scalery.inverse_transform(y_train2),bootstrap_means2_array.mean(0))))

# Define file names for training and testing data
train_file = 'train_data_output_final.xlsx'
test_file = 'test_data_output_final.xlsx'

# Save the training data to an Excel file
train_data_output_final = pd.concat([X_train, y_train], axis=1)
train_data_output_final.to_excel(train_file, index=False)

# Save the testing data to an Excel file
test_data_output_final = pd.concat([X_test, y_test], axis=1)
test_data_output_final.to_excel(test_file, index=False)

# Add predictions to the 'test' DataFrame
test_data_output_final['Predicted_Wrec'] = bootstrap_means_array.mean(0)
test_data_output_final['Error'] = bootstrap_means_array.std(0)

# Add predictions to the 'train' DataFrame
train_data_output_final['Predicted_Wrec'] = bootstrap_means2_array.mean(0)
train_data_output_final['Error'] = bootstrap_means2_array.std(0)

# Save the updated DataFrame with predictions to a new file
output_file = 'test_predicted_Wrec-new-feature.xlsx.'  # Update with the desired output file path
test_data_output_final.to_excel('test_predicted_Wrec-new-feature.xlsx', index=False)

# Save the updated DataFrame with predictions to a new file
output_file = 'train_predicted_Wrec-new-feature.xlsx.'  # Update with the desired output file path
train_data_output_final.to_excel('train_predicted_Wrec-new-feature.xlsx', index=False)

In [ ]:
GS11=XGBRegressor(n_estimators=800,max_depth=3,eta=0.01)
GS11.fit(X_train1,y_train1)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eta=0.01, eval_metric=None,
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=800, n_jobs=None, ...)

# **Features Importance**

In [ ]:
from matplotlib import pyplot
from xgboost import XGBClassifier
from xgboost import plot_importance
import pandas as pd
dictionary={}
for col,score in zip(train.columns,GS11.feature_importances_):
    print(col,score)
    dictionary[col]=score


print(GS11.feature_importances_)
feature_important = model_xgb.get_booster().get_score(importance_type='weight')
keys = list(feature_important.keys())
values = list(feature_important.values())

data = pd.DataFrame(data=values, index=keys, columns=["score"]).sort_values(by = "score", ascending=True)

In [ ]:
from collections import OrderedDict
import numpy as np

keys = list(dictionary.keys())
values = list(dictionary.values())
sorted_value_index = np.argsort(values)
sorted_dict = {keys[i]: values[i] for i in sorted_value_index}

print(sorted_dict)